# 06. Model Validation

This notebook covers the sixth step in a typical QSAR workflow:
- Internal validation using cross-validation
- External validation on the independent test set
- Calculating performance metrics (R², RMSE, MAE, Q²)
- Visualizing prediction quality

Model validation is crucial to assess the predictive performance and robustness of QSAR models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import cross_val_predict, KFold
from scipy.stats import pearsonr

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 6.1 Load Data and Models

Load the training and test datasets, and the trained model.

In [ ]:
# Load training and test data
train_path = '../Project/train_data.csv'
test_path = '../Project/test_data.csv'

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print(f"Training data: {df_train.shape}")
print(f"Test data: {df_test.shape}")

In [ ]:
# Prepare data
feature_cols = [col for col in df_train.columns if col not in ['Smiles', 'pChEMBL']]

X_train = df_train[feature_cols].values
y_train = df_train['pChEMBL'].values
X_test = df_test[feature_cols].values
y_test = df_test['pChEMBL'].values

print(f"\nTraining set: {X_train.shape}, {y_train.shape}")
print(f"Test set: {X_test.shape}, {y_test.shape}")

In [ ]:
# Load the best trained model
model_path = '../Project/models/best_model.pkl'

with open(model_path, 'rb') as f:
    model = pickle.load(f)

print(f"Model loaded: {type(model).__name__}")

## 6.2 Internal Validation (Cross-Validation)

Perform k-fold cross-validation on the training set to assess internal consistency.

In [ ]:
# Perform cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

print("Performing 5-fold cross-validation...")
y_train_pred_cv = cross_val_predict(model, X_train, y_train, cv=cv, n_jobs=-1)

# Calculate CV metrics
r2_cv = r2_score(y_train, y_train_pred_cv)
rmse_cv = np.sqrt(mean_squared_error(y_train, y_train_pred_cv))
mae_cv = mean_absolute_error(y_train, y_train_pred_cv)
q2_cv = r2_cv  # Q² is essentially the cross-validated R²

print(f"\nCross-Validation Results:")
print(f"  Q² (R² CV): {q2_cv:.4f}")
print(f"  RMSE CV: {rmse_cv:.4f}")
print(f"  MAE CV: {mae_cv:.4f}")

In [ ]:
# Plot cross-validation results
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Predicted vs Actual (CV)
axes[0].scatter(y_train, y_train_pred_cv, alpha=0.6, s=50)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 
             'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual pChEMBL')
axes[0].set_ylabel('Predicted pChEMBL (CV)')
axes[0].set_title(f'Cross-Validation Results\nQ² = {q2_cv:.4f}, RMSE = {rmse_cv:.4f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residual plot (CV)
residuals_cv = y_train - y_train_pred_cv
axes[1].scatter(y_train_pred_cv, residuals_cv, alpha=0.6, s=50)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted pChEMBL (CV)')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot (Cross-Validation)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6.3 Training Set Performance

Evaluate the model on the full training set.

In [ ]:
# Predict on training set
y_train_pred = model.predict(X_train)

# Calculate training metrics
r2_train = r2_score(y_train, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
mae_train = mean_absolute_error(y_train, y_train_pred)
pearson_r_train, pearson_p_train = pearsonr(y_train, y_train_pred)

print(f"Training Set Performance:")
print(f"  R²: {r2_train:.4f}")
print(f"  RMSE: {rmse_train:.4f}")
print(f"  MAE: {mae_train:.4f}")
print(f"  Pearson r: {pearson_r_train:.4f} (p = {pearson_p_train:.2e})")

## 6.4 External Validation (Test Set)

Evaluate the model on the independent test set.

In [ ]:
# Predict on test set
y_test_pred = model.predict(X_test)

# Calculate test metrics
r2_test = r2_score(y_test, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae_test = mean_absolute_error(y_test, y_test_pred)
pearson_r_test, pearson_p_test = pearsonr(y_test, y_test_pred)

print(f"Test Set Performance:")
print(f"  R²: {r2_test:.4f}")
print(f"  RMSE: {rmse_test:.4f}")
print(f"  MAE: {mae_test:.4f}")
print(f"  Pearson r: {pearson_r_test:.4f} (p = {pearson_p_test:.2e})")

In [ ]:
# Plot test set results
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Predicted vs Actual (Test)
axes[0].scatter(y_test, y_test_pred, alpha=0.6, s=50, color='green')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual pChEMBL')
axes[0].set_ylabel('Predicted pChEMBL')
axes[0].set_title(f'Test Set Performance\nR² = {r2_test:.4f}, RMSE = {rmse_test:.4f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residual plot (Test)
residuals_test = y_test - y_test_pred
axes[1].scatter(y_test_pred, residuals_test, alpha=0.6, s=50, color='green')
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted pChEMBL')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot (Test Set)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6.5 Comprehensive Performance Comparison

Compare performance across training, CV, and test sets.

In [ ]:
# Create comparison table
performance_comparison = pd.DataFrame({
    'Dataset': ['Training', 'Cross-Validation', 'Test'],
    'R²': [r2_train, q2_cv, r2_test],
    'RMSE': [rmse_train, rmse_cv, rmse_test],
    'MAE': [mae_train, mae_cv, mae_test],
    'Pearson r': [pearson_r_train, pearson_r_train, pearson_r_test]
})

print("\nPerformance Comparison:")
print(performance_comparison.to_string(index=False))

# Save performance comparison
performance_comparison.to_csv('../Project/performance_comparison.csv', index=False)
print("\nPerformance comparison saved to: ../Project/performance_comparison.csv")

In [ ]:
# Visualize performance comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

datasets = performance_comparison['Dataset']
x_pos = np.arange(len(datasets))

# R² comparison
axes[0].bar(x_pos, performance_comparison['R²'], color=['blue', 'orange', 'green'], alpha=0.7)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(datasets)
axes[0].set_ylabel('R² Score')
axes[0].set_title('R² Comparison')
axes[0].set_ylim([0, 1])
axes[0].grid(True, alpha=0.3)

# RMSE comparison
axes[1].bar(x_pos, performance_comparison['RMSE'], color=['blue', 'orange', 'green'], alpha=0.7)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(datasets)
axes[1].set_ylabel('RMSE')
axes[1].set_title('RMSE Comparison')
axes[1].grid(True, alpha=0.3)

# MAE comparison
axes[2].bar(x_pos, performance_comparison['MAE'], color=['blue', 'orange', 'green'], alpha=0.7)
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(datasets)
axes[2].set_ylabel('MAE')
axes[2].set_title('MAE Comparison')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6.6 Prediction Analysis

Analyze the quality and distribution of predictions.

In [ ]:
# Combine predictions for visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Training set: Predicted vs Actual
axes[0, 0].scatter(y_train, y_train_pred, alpha=0.5, s=30)
axes[0, 0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual pChEMBL')
axes[0, 0].set_ylabel('Predicted pChEMBL')
axes[0, 0].set_title(f'Training Set (R² = {r2_train:.4f})')
axes[0, 0].grid(True, alpha=0.3)

# Test set: Predicted vs Actual
axes[0, 1].scatter(y_test, y_test_pred, alpha=0.5, s=30, color='green')
axes[0, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 1].set_xlabel('Actual pChEMBL')
axes[0, 1].set_ylabel('Predicted pChEMBL')
axes[0, 1].set_title(f'Test Set (R² = {r2_test:.4f})')
axes[0, 1].grid(True, alpha=0.3)

# Residuals distribution (Training)
axes[1, 0].hist(y_train - y_train_pred, bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].axvline(x=0, color='r', linestyle='--', lw=2)
axes[1, 0].set_xlabel('Residuals')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title(f'Training Set Residuals (MAE = {mae_train:.4f})')
axes[1, 0].grid(True, alpha=0.3)

# Residuals distribution (Test)
axes[1, 1].hist(y_test - y_test_pred, bins=20, edgecolor='black', alpha=0.7, color='green')
axes[1, 1].axvline(x=0, color='r', linestyle='--', lw=2)
axes[1, 1].set_xlabel('Residuals')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title(f'Test Set Residuals (MAE = {mae_test:.4f})')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6.7 Save Predictions

Save predictions for further analysis.

In [ ]:
# Save test set predictions
test_predictions = df_test[['Smiles', 'pChEMBL']].copy()
test_predictions['pChEMBL_predicted'] = y_test_pred
test_predictions['residual'] = y_test - y_test_pred
test_predictions['absolute_error'] = np.abs(test_predictions['residual'])

test_predictions.to_csv('../Project/test_predictions.csv', index=False)
print(f"Test predictions saved to: ../Project/test_predictions.csv")

# Display worst predictions
print("\nWorst 5 predictions (highest absolute error):")
print(test_predictions.nlargest(5, 'absolute_error')[['Smiles', 'pChEMBL', 'pChEMBL_predicted', 'absolute_error']])

## 6.8 Summary

Summarize the model validation results.

In [ ]:
print("="*70)
print("MODEL VALIDATION SUMMARY")
print("="*70)
print(f"Model: {type(model).__name__}")
print(f"\nTraining Set ({len(y_train)} compounds):")
print(f"  R²: {r2_train:.4f}")
print(f"  RMSE: {rmse_train:.4f}")
print(f"  MAE: {mae_train:.4f}")
print(f"\nCross-Validation (5-fold):")
print(f"  Q²: {q2_cv:.4f}")
print(f"  RMSE CV: {rmse_cv:.4f}")
print(f"  MAE CV: {mae_cv:.4f}")
print(f"\nTest Set ({len(y_test)} compounds):")
print(f"  R²: {r2_test:.4f}")
print(f"  RMSE: {rmse_test:.4f}")
print(f"  MAE: {mae_test:.4f}")
print(f"  Pearson r: {pearson_r_test:.4f}")

# Model robustness check
if abs(r2_train - r2_test) < 0.2:
    print("\n✓ Model shows good generalization (training and test R² are similar)")
else:
    print("\n⚠ Model may be overfitting (large gap between training and test R²)")

if q2_cv > 0.5:
    print("✓ Model has good internal predictive ability (Q² > 0.5)")
else:
    print("⚠ Model has weak internal predictive ability (Q² < 0.5)")

if r2_test > 0.6:
    print("✓ Model has good external predictive ability (R²_test > 0.6)")
else:
    print("⚠ Model has moderate external predictive ability (R²_test < 0.6)")

print(f"\nPredictions saved to: ../Project/test_predictions.csv")
print("\nModel validation complete!")
print("="*70)

## Next Steps

The model has been validated on both internal and external datasets. The next notebook (07_applicability_domain.ipynb) will:
- Evaluate the applicability domain of the model
- Determine the chemical space where predictions are reliable
- Visualize AD boundaries and identify outliers